# Challenge — Red neuronal con Keras 3
### Universidad EAFIT | SI3003 — Introducción a la Inteligencia Artificial

---

## Objetivo

En clase construimos una red neuronal para clasificación de imágenes usando **Keras 3**.

En este ejercicio aplicarás **la misma receta** sobre un dataset diferente: **CIFAR-10 convertido a escala de grises**.

No necesitas diseñar un pipeline nuevo. La descarga, conversión a escala de grises y visualización están resueltas. Tu trabajo es completar únicamente las etapas de Keras que vimos en clase:

```text
Datos → Normalización → Modelo → Loss + Optimizer → fit() → evaluate() → Predicción
```

Las celdas marcadas con `# TODO` son las que debes completar.


---
## 0. Librerías


In [ ]:
import keras
from keras import layers
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
keras.utils.set_random_seed(SEED)

print(f"Keras version : {keras.__version__}")
print(f"Backend       : {keras.backend.backend()}")


---
# 1. Dataset: CIFAR-10

CIFAR-10 contiene 50,000 imágenes de entrenamiento y 10,000 de prueba. Las imágenes originales son de **32×32×3** y pertenecen a 10 categorías:

`airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck`.


In [ ]:
# Esta parte está resuelta.
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

y_train = y_train.squeeze()
y_test = y_test.squeeze()

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("X_train original:", X_train.shape)
print("X_test original :", X_test.shape)
print("y_train         :", y_train.shape)


## 1.1 RGB → escala de grises

Usaremos la transformación:

$$Gray = 0.299R + 0.587G + 0.114B$$

Esta parte está resuelta porque el objetivo del ejercicio es practicar Keras.


In [ ]:
X_train = (
    0.299 * X_train[..., 0] +
    0.587 * X_train[..., 1] +
    0.114 * X_train[..., 2]
)

X_test = (
    0.299 * X_test[..., 0] +
    0.587 * X_test[..., 1] +
    0.114 * X_test[..., 2]
)

print("X_train grayscale:", X_train.shape)
print("X_test grayscale :", X_test.shape)


### Pregunta 1

Después de convertir las imágenes a escala de grises, cada imagen tiene tamaño `32×32`.

¿Cuántos valores tendrá cada imagen después de aplicar `Flatten()`?

**Respuesta:** 1024 valores. Cada imagen en escala de grises es de 32×32, y `Flatten()` convierte esa matriz en un vector unidimensional: 32 × 32 = 1024. (Si fuera RGB serían 32 × 32 × 3 = 3072, pero al pasar a grises se elimina el canal de color.)


## 1.2 Normalización

Los valores de los píxeles están entre 0 y 255. Completa el código para llevarlos al rango `[0,1]`.


In [ ]:
# TODO 1 — Normalización
X_train = X_train.astype("float32") / 255.0
X_test  = X_test.astype("float32") / 255.0

print(X_train.min(), X_train.max())


In [ ]:
# Validación TODO 1
assert X_train.dtype == np.float32
assert X_test.dtype == np.float32
assert X_train.min() >= 0.0
assert X_train.max() <= 1.0
print("✓ Datos normalizados correctamente")


## 1.3 Visualización

Esta parte está resuelta.


In [ ]:
plt.figure(figsize=(12, 6))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_train[i], cmap="gray")
    plt.title(class_names[y_train[i]])
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()


---
# 2. Construir la red neuronal

Construiremos la misma arquitectura conceptual vista en clase:

```text
Imagen 32×32
   ↓
Flatten
   ↓
1024 valores
   ↓
Dense(128)
   ↓
ReLU
   ↓
Dense(10)
   ↓
Logits
```

> La última capa **no** debe tener Softmax. Trabajaremos con logits.


In [ ]:
# TODO 2 — Completa la arquitectura
model = keras.Sequential([
    keras.Input(shape=(32, 32)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10)
])

model.summary()


In [ ]:
# Validación TODO 2
expected_params = (1024 * 128 + 128) + (128 * 10 + 10)
assert model.count_params() == expected_params
print(f"✓ Arquitectura correcta: {model.count_params():,} parámetros")


### Preguntas

**2.** ¿Por qué necesitamos `Flatten()` antes de la primera capa `Dense`?

**Respuesta:** Una capa `Dense` (totalmente conectada) espera como entrada un vector 1D por cada ejemplo, no una matriz 2D. La imagen tiene forma 32×32, así que `Flatten()` la reorganiza en un vector de 1024 valores para que cada píxel se conecte con las neuronas de la capa densa. Sin aplanar, las dimensiones no serían compatibles con la multiplicación matricial que hace `Dense`.


**3.** ¿Por qué la última capa tiene 10 neuronas?

**Respuesta:** Porque CIFAR-10 tiene 10 clases (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck). Cada neurona de salida produce un logit (puntuación) asociado a una de esas clases, de modo que hay una salida por categoría posible.


**4.** ¿Qué función cumple `ReLU`?

**Respuesta:** `ReLU` (Rectified Linear Unit, f(x) = max(0, x)) introduce **no linealidad** en la red. Sin una función de activación no lineal, apilar capas densas equivaldría a una sola transformación lineal y la red no podría aprender relaciones complejas. Además es eficiente de calcular y ayuda a mitigar el problema del desvanecimiento del gradiente respecto a activaciones como sigmoide o tanh.



---
# 3. Configurar el entrenamiento

Usaremos:

- `Adam`
- `SparseCategoricalCrossentropy`
- `accuracy`

Como la red produce logits, la pérdida debe configurarse con `from_logits=True`.


In [ ]:
lr_rate = 0.001

# TODO 3 — Configura el entrenamiento
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_rate),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)


### Pregunta 5

¿Por qué usamos `from_logits=True`?

**Respuesta:** Porque la última capa de la red **no** aplica `Softmax`: produce logits (puntuaciones sin normalizar). Al indicar `from_logits=True`, la función de pérdida `SparseCategoricalCrossentropy` aplica internamente el `log-softmax` sobre esos logits. Esto es numéricamente más estable (evita calcular exp/log por separado, que puede causar overflow/underflow) que aplicar Softmax en la capa y luego el logaritmo en la pérdida.


---
# 4. Entrenar el modelo

Usaremos 10 épocas y mini-batches de 64 ejemplos. Completa `fit()`.


In [ ]:
epochs = 10
batch_size = 64

# TODO 4 — Entrenamiento
history = model.fit(
    X_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test, y_test),
    shuffle=True
)


### Preguntas

**6.** ¿Qué representa una `epoch`?

**Respuesta:** Una época es una pasada completa del algoritmo sobre **todo** el conjunto de entrenamiento (las 50,000 imágenes). En cada época el modelo ve una vez cada ejemplo y actualiza sus pesos. Aquí se entrena durante 10 épocas, es decir, el dataset completo se recorre 10 veces.


**7.** ¿Qué representa `batch_size=64`?

**Respuesta:** Es el número de ejemplos que se procesan juntos antes de calcular el gradiente y actualizar los pesos una vez. En lugar de usar todo el dataset (batch completo) o un solo ejemplo (SGD puro), se usan mini-lotes de 64 imágenes. Con 50,000 imágenes y lotes de 64, cada época realiza aproximadamente 782 actualizaciones de pesos (⌈50000/64⌉). Los mini-batches equilibran estabilidad del gradiente y eficiencia de cómputo.



## 4.1 Curvas de aprendizaje

Esta parte está resuelta.


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Evolución del entrenamiento")
plt.legend()
plt.show()


### Pregunta 8

Observa las dos curvas. ¿Hay evidencia de overfitting? Justifica brevemente.

**Respuesta:** Sí, típicamente se observa overfitting en este modelo. El indicio es la **brecha creciente** entre la curva de train accuracy y la de validation accuracy: la precisión de entrenamiento sigue subiendo época tras época mientras que la de validación se estanca (o incluso empieza a bajar). Cuando la red se ajusta cada vez mejor a los datos de entrenamiento pero deja de mejorar (o empeora) en los de validación, está memorizando el ruido del conjunto de entrenamiento en lugar de generalizar. Con una red densa sobre imágenes 32×32 y solo escala de grises esta separación suele aparecer en las últimas épocas. (Confirma la magnitud exacta mirando tus curvas al ejecutar.)


---
# 5. Evaluar el modelo

Completa `evaluate()`.


In [ ]:
# TODO 5 — Evaluación
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    batch_size=batch_size,
    verbose=0
)

print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_accuracy:.4f}")
print(f"Accuracy (%)  : {test_accuracy * 100:.2f}%")


### Pregunta 9

Compara este resultado con Fashion-MNIST, utilizado en clase.

¿CIFAR-10 en escala de grises parece más fácil o más difícil para esta red? Justifica usando el accuracy obtenido.

**Respuesta:** Mucho más **difícil**. Con esta misma arquitectura densa, Fashion-MNIST alcanza típicamente ~88–90% de accuracy, mientras que CIFAR-10 en escala de grises con esta red suele quedarse alrededor del 40–45% (aún muy por encima del 10% del azar, pero bajo). Las razones: (1) los objetos de CIFAR-10 son fotografías naturales con fondos complejos, distintas poses, iluminación y escalas, mientras que Fashion-MNIST tiene prendas centradas sobre fondo uniforme; (2) al pasar a escala de grises se pierde el color, una pista muy informativa en CIFAR-10 (p. ej. cielo azul de un avión, verde de una rana); y (3) una red densa ignora la estructura espacial. Todo esto hace que el accuracy sea claramente inferior al de Fashion-MNIST. (Sustituye por tu valor exacto tras ejecutar.)


---
# 6. Logits y Softmax

La salida de la red son 10 logits. Para interpretarlos como probabilidades necesitamos Softmax.


In [ ]:
index = 0
image = X_test[index]
true_class = y_test[index]

plt.imshow(image, cmap="gray")
plt.title(f"Clase real: {class_names[true_class]}")
plt.axis("off")
plt.show()

image_batch = np.expand_dims(image, axis=0)
logits = model(image_batch, training=False)

print("Logits:")
print(keras.ops.convert_to_numpy(logits))


In [ ]:
# TODO 6 — Logits -> probabilidades
probabilities = keras.ops.softmax(logits)

# TODO 7 — Clase con mayor probabilidad
prediction = keras.ops.argmax(probabilities, axis=1)
prediction = int(keras.ops.convert_to_numpy(prediction)[0])

print("Clase real     :", class_names[true_class])
print("Clase predicha :", class_names[prediction])


In [ ]:
probabilities_np = keras.ops.convert_to_numpy(probabilities)[0]

print("\nProbabilidades:")
for i, p in enumerate(probabilities_np):
    marker = " ← predicción" if i == prediction else ""
    print(f"{class_names[i]:12s}: {p:.4f}{marker}")

print("\nSuma:", probabilities_np.sum())


### Pregunta 10

¿Por qué las probabilidades producidas por `Softmax` deben sumar aproximadamente 1?

**Respuesta:** Porque `Softmax` está diseñada precisamente para producir una **distribución de probabilidad** sobre las clases. Toma los logits, los exponencia (e^{z_i}, siempre positivo) y divide cada uno entre la suma de todas las exponenciales: p_i = e^{z_i} / Σ_j e^{z_j}. Al dividir por la suma total, los valores quedan entre 0 y 1 y su suma es exactamente 1 (la pequeña desviación que se ve, como 0.9999…, es solo error de redondeo en coma flotante). Esto permite interpretar cada salida como la probabilidad de que la imagen pertenezca a esa clase, siendo las 10 clases mutuamente excluyentes y exhaustivas.


---
# 7. Varias predicciones

Esta parte está resuelta. Observa especialmente los errores del modelo.


In [ ]:
n = 12
logits_batch = model(X_test[:n], training=False)
predictions = keras.ops.argmax(logits_batch, axis=1)
predictions = keras.ops.convert_to_numpy(predictions)

plt.figure(figsize=(12, 8))
for i in range(n):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_test[i], cmap="gray")
    real = class_names[y_test[i]]
    pred = class_names[predictions[i]]
    plt.title(f"Real: {real}\nPred: {pred}", fontsize=9)
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()


---
# 8. Reflexión final

### 1. ¿Cuál fue el accuracy final?

**Respuesta:** El accuracy de prueba (test) obtenido con esta red densa sobre CIFAR-10 en escala de grises se sitúa aproximadamente entre **40% y 45%**. Reemplaza este valor por el número exacto que imprime la celda de `evaluate()` al ejecutar el notebook. Aunque es bajo, está muy por encima del 10% que daría adivinar al azar entre 10 clases.


### 2. ¿Qué clases parecen más difíciles de distinguir?

**Respuesta:** Las categorías de **animales** suelen confundirse entre sí, especialmente `cat` (gato) vs `dog` (perro), y también `deer` (venado) vs `horse` (caballo), o `bird` (pájaro) vs `airplane` (avión). Comparten siluetas, texturas y fondos naturales parecidos, y al quitar el color se pierde una pista clave que ayudaría a separarlas. En cambio, clases con formas muy definidas como `automobile`, `truck` o `ship` suelen ser más fáciles.


### 3. Después de `Flatten()`, la imagen se convierte en un vector de 1024 valores. ¿Qué información importante sobre la imagen podría perderse al tratarla únicamente como un vector?

**Respuesta:** Se pierde la **estructura espacial** de la imagen: la relación de vecindad entre píxeles. Al aplanar, dos píxeles que están uno al lado del otro (arriba/abajo, izquierda/derecha) quedan tan "lejos" en el vector como dos píxeles en esquinas opuestas. La red densa deja de saber qué píxeles son adyacentes, por lo que no puede aprovechar patrones locales como bordes, esquinas, texturas o formas, ni la invariancia a traslaciones (que un objeto siga siendo el mismo aunque se mueva unos píxeles).


### 4. ¿Qué tipo de arquitectura podría aprovechar mejor la estructura espacial de las imágenes?

**Respuesta:** Las **Redes Neuronales Convolucionales (CNNs)**. Sus capas convolucionales aplican filtros que se deslizan sobre la imagen y detectan patrones locales (bordes, texturas, formas) preservando la disposición 2D de los píxeles. Gracias al peso compartido y al pooling logran invariancia a traslaciones y aprenden jerarquías de características, obteniendo un accuracy mucho mayor que una red densa en imágenes como CIFAR-10.

---

## Para pensar

Nuestro modelo hace:

```text
32×32 → Flatten → 1024 valores → Dense → ReLU → Dense(10)
```

Sin embargo, una imagen tiene **estructura espacial**: existen bordes, formas y patrones locales.

Este resultado nos deja preparada la siguiente pregunta del curso:

> **¿Cómo puede una red aprovechar directamente la estructura espacial de una imagen?**

La respuesta será: **Convolutional Neural Networks (CNNs)**.
